In [42]:

import numpy as np
import pandas as pd
import joblib

from scipy import stats

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder,StandardScaler,FunctionTransformer

from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.compose import ColumnTransformer

from sklearn.metrics.pairwise import rbf_kernel

from sklearn.tree import DecisionTreeRegressor

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,root_mean_squared_error


# Load Database

In [43]:

housing = pd.read_csv(
    "california_housing_prices.csv"
)

# Stratified Train / Test Split

In [44]:

housing["income_category"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

train_set, test_set = train_test_split(
    housing,
    test_size=0.2,
    random_state=40,
    stratify=housing["income_category"]
)

train_set = train_set.drop(columns="income_category")
test_set = test_set.drop(columns="income_category")


# Separate Features & Target


In [45]:

train_features = train_set.drop(
    columns="median_house_value"
)

train_target = train_set["median_house_value"]

test_features = test_set.drop(
    columns="median_house_value"
)

test_target = test_set["median_house_value"]


# Feature Engineering Functions

In [46]:

def ratio_of_columns(X):
    return X[:, [0]] / X[:, [1]]


def ratio_feature_name(transformer, feature_names_in):
    return ["ratio"]



# Feature Engineering Pipelines

In [47]:

def ratio_pipeline():

    return make_pipeline(

        SimpleImputer(strategy="median"),

        FunctionTransformer(
            ratio_of_columns,
            feature_names_out=ratio_feature_name
        ),

        StandardScaler()

    )


heavy_tail_pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    FunctionTransformer(
        np.log,
        feature_names_out="one-to-one"
    ),

    StandardScaler()

)

housing_age_pipeline = FunctionTransformer(

    rbf_kernel,

    feature_names_out="one-to-one",

    kw_args={
        "Y": [[35]],
        "gamma": 0.1
    }

)

default_numeric_pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    StandardScaler()

)

categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder()
    )

])

In [48]:

all_transformations = ColumnTransformer(

    [

        (
            "bedrooms_per_room",
            ratio_pipeline(),
            ["total_bedrooms", "total_rooms"]
        ),

        (
            "rooms_per_house",
            ratio_pipeline(),
            ["total_rooms", "households"]
        ),

        (
            "people_per_house",
            ratio_pipeline(),
            ["population", "households"]
        ),

        (
            "log_features",
            heavy_tail_pipeline,
            [
                "total_bedrooms",
                "total_rooms",
                "population",
                "households",
                "median_income"
            ]
        ),

        (
            "housing_age_rbf",
            housing_age_pipeline,
            ["housing_median_age"]
        ),

        (
            "categorical",
            categorical_pipeline,
            ["ocean_proximity"]
        )

    ],

    remainder=default_numeric_pipeline

)


# Transform Training Data

In [49]:

train_features_transformed = all_transformations.fit_transform(
    train_features
)


# Feature Names

In [50]:
feature_names = all_transformations.get_feature_names_out()

# Linear Regression Pipeline

In [51]:

linear_regression_pipeline = Pipeline(

    [

        (
            "preprocessing",
            all_transformations
        ),

        (
            "model",
            LinearRegression()
        )

    ]

)

# Decision Tree Pipeline

In [52]:

decision_tree_pipeline = Pipeline(

    [

        (
            "preprocessing",
            all_transformations
        ),

        (
            "model",
            DecisionTreeRegressor(
                random_state=40
            )
        )

    ]

)

# Train Linear Regression

In [53]:

linear_regression_pipeline.fit(

    train_features,
    train_target

)

linear_predictions = linear_regression_pipeline.predict(
    train_features
)


# Linear Regression Metrics

In [54]:


linear_mae = mean_absolute_error(
    train_target,
    linear_predictions
)

linear_rmse = mean_squared_error(
    train_target,
    linear_predictions,
   )

linear_root = root_mean_squared_error(
    train_target,
    linear_predictions,
   )

linear_r2 = r2_score(
    train_target,
    linear_predictions
)

print("=" * 60)
print("Linear Regression")
print("=" * 60)

print(f"MAE  : {linear_mae:.2f}")
print(f"RMSE : {linear_rmse:.2f}")
print(f"root : {linear_root:.2f}")
print(f"R²   : {linear_r2:.4f}")


Linear Regression
MAE  : 53721.06
RMSE : 5169319861.14
root : 71897.98
R²   : 0.6097


# Cross Validation

In [55]:

linear_cv = -cross_val_score(

    linear_regression_pipeline,

    train_features,

    train_target,

    scoring="neg_mean_absolute_error",

    cv=5

)

print("\nCross Validation (Linear Regression)")
print(pd.Series(linear_cv).describe())



Cross Validation (Linear Regression)
count        5.000000
mean     53858.315912
std        745.455015
min      52921.895396
25%      53293.868044
50%      54107.355312
75%      54182.759503
max      54785.701308
dtype: float64


# Train Decision Tree

In [56]:

decision_tree_pipeline.fit(

    train_features,
    train_target

)

decision_tree_predictions = decision_tree_pipeline.predict(
    train_features
)

# Decision Tree Metrics


In [57]:

decision_tree_mae = mean_absolute_error(
    train_target,
    decision_tree_predictions
)

decision_tree_rmse = mean_squared_error(
    train_target,
    decision_tree_predictions
)

decision_tree_root = root_mean_squared_error(
    train_target,
    decision_tree_predictions
)

decision_tree_r2 = r2_score(
    train_target,
    decision_tree_predictions
)

print("\n" + "=" * 60)
print("Decision Tree")
print("=" * 60)

print(f"MAE  : {decision_tree_mae:.2f}")
print(f"RMSE : {decision_tree_rmse:.2f}")
print(f"root : {decision_tree_root:.2f}")
print(f"R²   : {decision_tree_r2:.4f}")



Decision Tree
MAE  : 0.00
RMSE : 0.00
root : 0.00
R²   : 1.0000


In [ ]:
# Cross Validation (Decision Tree)

In [58]:

decision_tree_cv = -cross_val_score(

    decision_tree_pipeline,

    train_features,

    train_target,

    scoring="neg_mean_absolute_error",

    cv=5

)

print("\nCross Validation (Decision Tree)")
print(pd.Series(decision_tree_cv).describe())



Cross Validation (Decision Tree)
count        5.000000
mean     46484.417362
std        683.185550
min      45682.993642
25%      45987.359673
50%      46421.758328
75%      47011.127498
max      47318.847668
dtype: float64
